# 06 · Website replay data (Colab, GPU)

Re-flies 150 landings of the harder experiment (random safe aim point, detector v2) so a full set of results exists on Drive, then re-flies a few instructive ones with camera frames recorded and exports them for the website (`web/public/data`). Needs the v2 checkpoint on Drive at `touchdown/runs/unet_v2/best.pt`. Approve the Google Drive popup in cell 2.

Replays are re-flown on this GPU, so tiny rendering differences from the Kaggle Monte Carlo are possible; the website says so.

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /content/touch_down || (cd /content/touch_down && git pull -q)
!sh /content/touch_down/scripts/colab_setup.sh 2>&1 | tail -3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/touchdown'
import os
CK = f'{D}/runs/unet_v2/best.pt'
assert os.path.getsize(CK) == 17520863, 'v2 checkpoint not found on Drive'
MC = f'{D}/montecarlo/random_safe_colab'

In [ ]:
%cd /content/touch_down
os.environ['TOUCHDOWN_DATA'] = '/content/data'
os.environ['TOUCHDOWN_BLENDER'] = '/content/blender/blender'
# 1. 150 landings, resumable (existing seeds are skipped)
!python -m touchdown.sim.montecarlo --out {MC} --seeds 0 150 --device GPU --checkpoint {CK} --aim-mode random_safe

In [ ]:
# 2. pick instructive seeds from those results, re-fly them with frames recorded, write the website data
!python -m touchdown.analysis.export_replays --out /content/web_data --mc {MC} --checkpoint {CK} --device GPU --aim-mode random_safe
!cd /content && zip -qr {D}/web_data.zip web_data && ls -la {D}/web_data.zip && du -sh /content/web_data